# Phase 2: Snapshot Triangle Build

Just the core mechanism: what it actually takes to turn transactions into a snapshot triangle. 

In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
pd.options.display.float_format = '{:,.2f}'.format

from snapshot_builder import CUTOFF, GRANULARITY_TO_STEP, ceil_period, load_visible, snapshot_grid, build_snapshot_triangle

## Display helper: render one claim back into triangle shape

In [2]:
# GRANULARITY_TO_STEP is imported from snapshot_builder
# and shared by both build_snapshot_triangle()'s safeguard and this display
# function. claim_as_triangle() itself stays local as it's a display/eyeball
# tool for this notebook, not core mechanism

def claim_as_triangle(snapshot_triangle, claim_no, value_col="future_paid",
                    truncate_at_settlement=True, granularity="quarterly"):
    """Render one claim's rows back into triangle shape (snapshot rows x age columns).

    granularity: "quarterly" (every period -- the native grid, and periods are
    already quarters), "yearly" (every 4th period = 12 months), or pass an int
    step directly. FILTERS an already-built triangle down to a coarser display
    cadence -- doesn't rebuild anything.

    Display-only. truncate_at_settlement=True (default) hides ages beyond the
    claim's own settlement. Never affects what's written; pass False to see the
    full, untruncated stored rows.

    The claim's settled/exit row -- and any other on-grid snapshot with zero
    surviving age columns at this cadence -- is always included as a row
    (every age column blank), never silently dropped.
    """
    step = GRANULARITY_TO_STEP.get(granularity, granularity) if isinstance(granularity, str) else granularity

    all_rows = snapshot_triangle[snapshot_triangle.claim_no == claim_no]
    observed_native = all_rows[all_rows.row_status == "observed"]
    settled_rows = all_rows[all_rows.row_status == "settled"]

    on_grid_snapshots = observed_native[observed_native.snapshot_period % step == 0]
    claim_rows = observed_native[
        (observed_native.snapshot_period % step == 0) & (observed_native.observation_age % step == 0)
    ]

    if truncate_at_settlement:
        first_settled_age = (
            claim_rows[claim_rows["is_settled_at_obs"].astype(bool)]
            .groupby("snapshot_period")["observation_age"].min()
        )
        cutoff_age = claim_rows["snapshot_period"].map(first_settled_age)
        claim_rows = claim_rows[cutoff_age.isna() | (claim_rows["observation_age"] <= cutoff_age)]

    paid_so_far = pd.concat([
        on_grid_snapshots.groupby("snapshot_period")["paid_to_date"].first(),
        settled_rows.set_index("snapshot_period")["paid_to_date"],
    ]).rename("paid_to_date")

    pivot = claim_rows.pivot(index="snapshot_period", columns="observation_age", values=value_col)
    if pivot.empty:
        result = paid_so_far.to_frame()
    else:
        result = pd.concat([paid_so_far, pivot], axis=1)
    return result.sort_index()

## Run it on the real data

In [3]:
df = pd.read_csv('../data/synthetic_transactions_with_covariates.csv')
snapshot_triangle = build_snapshot_triangle(df)

portfolio: 3624 claims
excluded (notified at or after the last grid point, 40): 301 (8.3%) -- no room left to show any development


snapshot triangle: 382485 rows (379840 observed, 2645 settled-exit), 3323 claims represented


In [4]:
snapshot_triangle.head(10)

,claim_no,snapshot_period,observation_age,observation_period,row_status,future_paid,paid_to_date,is_settled_at_obs
0,1,2,1.00,3.00,observed,0.00,0.00,False
1,1,2,2.00,4.00,observed,0.00,0.00,False
2,1,2,3.00,5.00,observed,0.00,0.00,False
3,1,2,4.00,6.00,observed,0.00,0.00,False
4,1,2,5.00,7.00,observed,0.00,0.00,False
5,1,2,6.00,8.00,observed,0.00,0.00,False
6,1,2,7.00,9.00,observed,"20,536.32",0.00,False
7,1,2,8.00,10.00,observed,"20,536.32",0.00,False
8,1,2,9.00,11.00,observed,"38,078.57",0.00,False
9,1,2,10.00,12.00,observed,"38,078.57",0.00,False


## Viewing a claim as a triangle

In [5]:
claim_as_triangle(snapshot_triangle, claim_no=1, granularity='quarterly')

,paid_to_date,1.00,2.00,3.00,4.00,5.00,6.00,7.00,8.00,9.00,10.00,11.00,12.00,13.00,14.00,15.00,16.00
snapshot_period,,,,,,,,,,,,,,,,,
2,0.00,0.00,0.00,0.00,0.00,0.00,0.00,"20,536.32","20,536.32","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","274,456.02","294,223.32"
3,0.00,0.00,0.00,0.00,0.00,0.00,"20,536.32","20,536.32","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","274,456.02","294,223.32",NaN
4,0.00,0.00,0.00,0.00,0.00,"20,536.32","20,536.32","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","274,456.02","294,223.32",NaN,NaN
5,0.00,0.00,0.00,0.00,"20,536.32","20,536.32","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","274,456.02","294,223.32",NaN,NaN,NaN
6,0.00,0.00,0.00,"20,536.32","20,536.32","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","274,456.02","294,223.32",NaN,NaN,NaN,NaN
7,0.00,0.00,"20,536.32","20,536.32","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","274,456.02","294,223.32",NaN,NaN,NaN,NaN,NaN
8,0.00,"20,536.32","20,536.32","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","38,078.57","274,456.02","294,223.32",NaN,NaN,NaN,NaN,NaN,NaN
9,"20,536.32",0.00,"17,542.25","17,542.25","17,542.25","17,542.25","17,542.25","17,542.25","253,919.70","273,687.00",NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,"20,536.32","17,542.25","17,542.25","17,542.25","17,542.25","17,542.25","17,542.25","253,919.70","273,687.00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
claim_as_triangle(snapshot_triangle, claim_no=1, granularity='yearly')

,paid_to_date,4.00,8.00,12.00,16.00
snapshot_period,,,,,
4,0.00,0.00,"38,078.57","38,078.57","294,223.32"
8,0.00,"38,078.57","38,078.57","294,223.32",NaN
12,"38,078.57",0.00,"256,144.75",NaN,NaN
16,"38,078.57","256,144.75",NaN,NaN,NaN
18,"294,223.32",NaN,NaN,NaN,NaN


In [7]:
# the exit row -- claim #1 settles at true period 19, and 19 is itself a grid point at quarterly granularity
snapshot_triangle[(snapshot_triangle.claim_no == 1) & (snapshot_triangle.row_status == 'settled')]

,claim_no,snapshot_period,observation_age,observation_period,row_status,future_paid,paid_to_date,is_settled_at_obs
488,1,18,NaN,NaN,settled,NaN,"294,223.32",NaN


## Row status breakdown across the whole portfolio

In [8]:
snapshot_triangle['row_status'].value_counts()

row_status
observed    379840
settled       2645
Name: count, dtype: int64